# R Language Guide — Part 2: Data Visualization & Variability

This notebook covers:
1. **Base R Plots** — pie charts, barplots
2. **ggplot2 Basics** — bar charts, histograms, boxplots, line plots
3. **Advanced ggplot2** — stacked/dodged/filled bars, multi-group boxplots, time series
4. **Variability Indexes** — range, IQR, variance, standard deviation, coefficient of variation
5. **Gini Index** — concentration measure for categorical variables
6. **dplyr** — grouped summaries with pipes
7. **Maps** — plotting geographic data

---
## 1. BASE R PLOTS

First, let's load the tonni (tuna) dataset and compute frequency distributions.

In [ ]:
# Load data
dati <- read.csv("tonni.csv", sep = ";")
N <- dim(dati)[1]

# Create classes for LUNGHEZZA
dati$LUNGHEZZA_CL <- cut(
  dati$LUNGHEZZA,
  breaks = c(10, 15, 20, 25, 30)
)

# Frequency distribution
ni <- table(dati$LUNGHEZZA_CL)
fi <- table(dati$LUNGHEZZA_CL) / N
Ni <- cumsum(ni)
Fi <- Ni / N

distr_freq_lung <- as.data.frame(cbind(ni, fi, Ni, Fi))
print(distr_freq_lung)

### 1.1 Pie Charts

Pie charts are generally **not recommended** for statistical visualization,  
but here's how to create them in base R.

In [ ]:
# Basic pie chart (no labels, no colors)
pie(distr_freq_lung$fi)

In [ ]:
# Better pie chart with title, colors, and labels
pie(distr_freq_lung$fi, 
    main = "Distribuzione delle classi di lunghezza",
    col = rainbow(length(distr_freq_lung$fi)),
    labels = rownames(distr_freq_lung))

In [ ]:
# Pie chart with percentage labels and custom colors
labels <- paste(rownames(distr_freq_lung), " ", distr_freq_lung$fi * 100, "%", sep = "")
colors <- c("pink", "pink2", "pink3", "pink4")

pie(distr_freq_lung$fi, 
    main = "Distribuzione delle classi di lunghezza",
    col = colors,
    labels = labels,
    radius = 1,
    clockwise = TRUE)

### 1.2 Barplots (Base R)

Barplots are a better alternative to pie charts for frequency data.

In [ ]:
# Basic barplot
barplot(distr_freq_lung$fi, 
        main = "Distribuzione delle classi di lunghezza",
        names.arg = rownames(distr_freq_lung))

In [ ]:
# Better barplot with axis labels and y-axis limits
barplot(distr_freq_lung$fi, 
        main = "Distribuzione delle classi di lunghezza",
        names.arg = rownames(distr_freq_lung),
        xlab = "Classi di lunghezza",
        ylab = "Frequenza relativa",
        ylim = c(0, max(distr_freq_lung$fi) * 1.2),
        col = "blue")

---
## 2. GGPLOT2 — THE GRAMMAR OF GRAPHICS

`ggplot2` is the most popular R package for data visualization.  
It uses a layered approach: data → aesthetics → geometries → scales → themes.

In [ ]:
library(ggplot2)

### 2.1 Bar Charts with ggplot2

`geom_bar()` with `stat = "count"` creates bar charts from raw data.

In [ ]:
# Simple bar chart of length classes
p <- ggplot(data = dati) +
  geom_bar(aes(x = LUNGHEZZA_CL), stat = "count", color = "black", fill = "blue") +
  labs(title = "Distribuzione delle classi di lunghezza", 
       x = "Classi di lunghezza", 
       y = "Frequenza") +
  scale_y_continuous(breaks = seq(0, 8, 1)) +
  theme_classic(base_size = 16)
print(p)

### 2.2 Stacked Bar Charts

By moving `fill` inside `aes()`, you can color bars by a second variable.

In [ ]:
# Stacked bar chart by SESSO
p <- ggplot(data = dati) +
  geom_bar(aes(x = LUNGHEZZA_CL,
               fill = SESSO), 
           stat = "count", 
           color = "black",
           position = "stack") +
  labs(title = "Distribuzione delle classi di lunghezza", 
       x = "Classi di lunghezza", 
       y = "Frequenza") +
  scale_y_continuous(breaks = seq(0, 8, 1)) +
  theme_classic(base_size = 16)
print(p)

### 2.3 Dodged (Side-by-Side) Bar Charts

Use `position = "dodge"` to place bars side by side.

In [ ]:
# Dodged bar chart by LOCALITA
p <- ggplot(data = dati) +
  geom_bar(aes(x = LUNGHEZZA_CL,
               fill = LOCALITa), 
           stat = "count", 
           color = "black",
           position = "dodge") +
  labs(title = "Distribuzione delle classi di lunghezza", 
       x = "Classi di lunghezza", 
       y = "Frequenza") +
  scale_y_continuous(breaks = seq(0, 8, 1)) +
  theme(legend.position = "bottom") +
  theme_classic(base_size = 16)
print(p)

### 2.4 Filled Bar Charts (Proportions)

Use `position = "fill"` to show proportions instead of counts.

In [ ]:
# Filled bar chart (proportions) by LOCALITA
p <- ggplot(data = dati) +
  geom_bar(aes(x = LUNGHEZZA_CL,
               fill = LOCALITa), 
           stat = "count", 
           color = "black",
           position = "fill") +
  labs(title = "Distribuzione delle classi di lunghezza", 
       x = "Classi di lunghezza", 
       y = "Proporzione") +
  scale_y_continuous(breaks = seq(0, 1, 0.2)) +
  theme(legend.position = "bottom") +
  theme_classic(base_size = 16)
print(p)

---
## 3. TIME SERIES PLOTS

Let's load the births/deaths dataset and create line plots.

In [ ]:
# Load nascite e morti (births and deaths) dataset
dati_ts <- read.csv("nascite e morti.csv", sep = ",")
str(dati_ts)

In [ ]:
# Simple column chart
p <- ggplot(data = dati_ts) +
    geom_col(aes(x = tempo, y = nascite))
print(p)

In [ ]:
# Add a line on top of the bars
p <- ggplot(data = dati_ts) +
    geom_col(aes(x = tempo, y = nascite)) +
    geom_line(aes(x = tempo, y = nascite), color = "red", lwd = 1.5)
print(p)

In [ ]:
# Dual line plot: births vs deaths with legend
p <- ggplot(data = dati_ts) +
    geom_line(aes(x = tempo, y = nascite, color = "Nascite"), lwd = 1.5) +
    geom_line(aes(x = tempo, y = morti, color = "Morti"), lwd = 1.5) + 
    geom_point(aes(x = tempo, y = nascite), color = "green2", size = 3) +
    geom_point(aes(x = tempo, y = morti), color = "red", size = 3) +
    labs(title = "Nascite e morti in Italia", x = "Anno", y = "Numero di persone") +
    scale_color_manual(name = "Legenda", 
                        values = c("Nascite" = "green2", "Morti" = "red"), 
                        breaks = c("Nascite", "Morti")) +
    theme_classic(base_size = 16)
print(p)

In [ ]:
# Enhanced version with data labels and custom x-axis breaks
p <- ggplot(data = dati_ts) +
    geom_line(aes(x = tempo, y = nascite, color = "Nascite"), lwd = 1.5) +
    geom_line(aes(x = tempo, y = morti, color = "Morti"), lwd = 1.5) + 
    geom_point(aes(x = tempo, y = nascite), color = "green2", size = 3) +
    geom_point(aes(x = tempo, y = morti), color = "red", size = 3) +
    labs(title = "Nascite e morti in Italia", x = "Anno", y = "Numero di persone") +
    scale_color_manual(name = "", values = c("Nascite" = "green2", "Morti" = "red"), 
                        breaks = c("Nascite", "Morti")) +
    geom_text(aes(x = tempo, y = nascite - 5, label = nascite)) +
    geom_text(aes(x = tempo, y = morti - 5, label = morti)) +
    scale_x_continuous(breaks = seq(2000, 2020, 10)) +
    theme_classic(base_size = 16)
print(p)

---
## 4. BOXPLOTS & VARIABILITY (diamonds dataset)

Let's use the built-in `diamonds` dataset from ggplot2.

In [ ]:
library(ggplot2)
data("diamonds")
head(diamonds)
summary(diamonds)

In [ ]:
# Attach for direct column access
attach(diamonds)

# Summary of price
summary(price)

### 4.1 Range and Interquartile Range

In [ ]:
# Range
max(price); min(price); max(price) - min(price)  # the range
range(price)

# Interquartile Range
IQR(price)

### 4.2 Scatter Plot with Quantile Lines

In [ ]:
# Scatter plot of price with quantile reference lines
ggplot() +
  geom_point(aes(x = seq(1, length(price)), y = price)) +
  geom_hline(yintercept = quantile(price), color = "red") +
  geom_label(aes(x = 50000, y = quantile(price)), label = quantile(price), color = "red")

### 4.3 Boxplots

Boxplots are the best way to visualize the distribution of a numeric variable.

In [ ]:
# Base R boxplot
boxplot(price)
abline(h = quantile(price), col = "red")

In [ ]:
# Multi boxplot: price by color (using tilde notation)
boxplot(price ~ color)

In [ ]:
# ggplot2 boxplot (single variable)
ggplot() +
  geom_boxplot(aes(y = price), color = 'blue') +
  geom_hline(yintercept = quantile(price), color = "red")

In [ ]:
# ggplot2 boxplot by color
ggplot() +
  geom_boxplot(aes(x = color, y = price), color = 'blue', fill = 'green') +
  geom_hline(yintercept = quantile(price), color = "red")

In [ ]:
# Multi-group boxplot: price by color, filled by cut
ggplot() +
  geom_boxplot(aes(x = color, y = price, fill = cut), color = 'blue') +
  geom_hline(yintercept = quantile(price), color = "red")

---
## 5. VARIABILITY INDEXES

### 5.1 Variance & Standard Deviation

Variance measures the average squared deviation from the mean.  
Standard deviation is the square root of variance (same unit as the data).

In [ ]:
# Manual variance calculation (population formula: divide by n)
mean_price <- mean(price)
n <- length(price)
sigma2 <- sum((price - mean_price)^2) / n
print(paste("Population variance:", sigma2))

sigma <- sqrt(sigma2)
print(paste("Population std dev:", sigma))

In [ ]:
# Built-in functions (use n-1 by default — sample formula)
var(price)   # sample variance
sd(price)    # sample standard deviation

### 5.2 Histogram

Histograms show the distribution shape of a numeric variable.

In [ ]:
# Histogram of price
ggplot() +
  geom_histogram(aes(x = price), binwidth = 500)

### 5.3 Coefficient of Variation (CV)

CV = (sd / mean) × 100  
It's a **unitless** measure of relative variability, useful for comparing  
variables with different units or scales.

In [ ]:
# Define CV function
cv <- function(x) {
    return(sd(x) / mean(x) * 100)
}

# Compare CV of price vs a standardized variable
z <- scale(price)  # standardized (mean=0, sd=1)

print(paste("CV of price:", cv(price)))
print(paste("CV of z:", cv(z)))

### 5.4 Grouped Variability with dplyr

The `dplyr` package provides the `%>%` (pipe) operator for chaining operations.  
Use `group_by()` + `summarise()` to compute statistics by group.

In [ ]:
# install.packages("dplyr")
library(dplyr)

In [ ]:
# Group by cut and compute mean, sd, and CV of price
diamonds %>%
  group_by(cut) %>%
  summarise(
    mean_price = mean(price),
    sd_price = sd(price),
    cv_price = cv(price)
  )

---
## 6. GINI INDEX

The **Gini index** measures concentration/heterogeneity for categorical variables.  
It ranges from 0 (all observations in one category) to 1 (uniform distribution).

In [ ]:
# Define Gini index function (normalized)
giniindex <- function(x) {
    ni <- table(x)
    fi <- ni / length(x)
    fi2 <- fi^2
    J <- length(table(x))

    gini <- 1 / sum(fi2)
    gini.normalized <- gini / ((J - 1) / J)
    return(gini.normalized)
}

In [ ]:
# Check distribution of color
table(color)

In [ ]:
# Compute Gini index for color
giniindex(color)

---
## 7. MAPS

R can plot geographic data using the `mapdata` package.

In [ ]:
# Install mapdata if needed
# install.packages("mapdata")
library(mapdata)

# Get Italy map data
italia <- map_data("italy")
str(italia)

In [ ]:
# Plot Italy map
p <- ggplot(data = italia) +
    geom_map(map = italia,
             mapping = aes(map_id = region, fill = group),
             color = "black") +
    scale_x_continuous(breaks = c(7, 18)) +
    scale_y_continuous(breaks = c(37, 47)) +
    theme_bw(base_size = 16)
print(p)

---
## Summary — Part 2

| Concept | Key Functions / Packages |
|---------|------------------------|
| Pie charts | `pie()` |
| Barplots (base) | `barplot()` |
| Bar charts (ggplot2) | `geom_bar()`, `position = "stack"/"dodge"/"fill"` |
| Line plots | `geom_line()`, `geom_point()` |
| Boxplots | `boxplot()`, `geom_boxplot()` |
| Histograms | `geom_histogram()` |
| Scatter plots | `geom_point()` |
| Range / IQR | `range()`, `IQR()`, `quantile()` |
| Variance / SD | `var()`, `sd()`, manual calculation |
| Coefficient of Variation | `cv <- function(x) sd(x)/mean(x)*100` |
| Grouped summaries | `dplyr::group_by()`, `summarise()`, `%>%` |
| Gini index | Custom function using `table()` |
| Maps | `mapdata::map_data()`, `geom_map()` |
| Themes | `theme_classic()`, `theme_bw()`, `labs()` |